# 7.6 補齊 batch 的位置該算 loss 嗎？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：補齊長度像空椅子：不是新客人**

attention 別讀 PAD，loss 也別考 PAD。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/padding.svg")))

**先想一想：**多補十個PAD會讓loss看起來更低嗎？

batch要補成相同長度，但PAD不是答案。把PAD標籤設爲ignored，再用有效token分母，添加填充不改變原本loss。

**把直覺寫成數學：**loss_sum/有效token數，而非除以所有B×T位置。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import masked_loss

a = torch.randn(1, 2, 4)
labels = torch.tensor([[1, 2]])
b = torch.cat([a, torch.randn(1, 3, 4)], 1)
extended = torch.tensor([[1, 2, -100, -100, -100]])
print(masked_loss(a, labels), masked_loss(b, extended))
assert torch.allclose(masked_loss(a, labels), masked_loss(b, extended))

**如何讀結果：**這節只檢查loss；7.7再保證PAD不會被當context。

**只改一件事：**只把一個PAD標籤改成詞表ID，檢查loss改變。
